# 03 - Alistamiento de datos | Pregunta 1

## Pregunta de negocio

¿En qué proveedores, modalidades y tipos de contrato se concentra la contratación de INVIAS, y existen patrones de concentración que requieran seguimiento por parte de la Dirección y los organismos de control?

## Objetivo del alistamiento

Preparar la base limpia de contratos de INVIAS para el análisis de concentración contractual.

En esta etapa se construyen:

- Variables temporales.
- Poblaciones para análisis por número de contratos y por valor contratado.
- Agregaciones por proveedor.
- Agregaciones por modalidad de contratación.
- Agregaciones por tipo de contrato.
- Cruces entre proveedor, modalidad y tipo de contrato.
- Indicadores descriptivos de concentración como CR1, CR5, CR10 y HHI.
- Bases auxiliares para las visualizaciones y el análisis posterior.

Esta etapa no busca interpretar todavía los resultados, sino dejar la información preparada para el análisis.

In [26]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Carga de la base limpia

In [27]:
# Cargamos la base obtenida en la etapa de limpieza

df = pd.read_csv(
    "df_limpio_p1.csv",
    low_memory=False
)

print("Base cargada correctamente.")
print("Registros:", df.shape[0])
print("Variables:", df.shape[1])

Base cargada correctamente.
Registros: 17477
Variables: 20


## 2. Preparación de variables

In [28]:
# Convertimos nuevamente la fecha de firma a formato fecha para evitar posibles errores

df["fecha_de_firma"] = pd.to_datetime(
    df["fecha_de_firma"],
    errors="coerce"
)

# Creamos variables temporales

df["anio_firma"] = df["fecha_de_firma"].dt.year.astype("Int64")
df["mes_firma"] = df["fecha_de_firma"].dt.month.astype("Int64")

df["periodo_firma"] = (
    df["fecha_de_firma"]
    .dt.to_period("M")
    .astype("string")
)

# Creamos variables monetarias para facilitar la interpretación

df["valor_millones"] = df["valor_del_contrato"] / 1_000_000
df["valor_miles_millones"] = df["valor_del_contrato"] / 1_000_000_000

df.head()

,id_contrato,referencia_del_contrato,proveedor_adjudicado,documento_proveedor,tipodocproveedor,valor_del_contrato,modalidad_de_contratacion,tipo_de_contrato,estado_contrato,fecha_de_firma,es_grupo,es_pyme,proveedor_normalizado,documento_proveedor_limpio,proveedor_id,proveedor_etiqueta,contrato_formalizado,valor_positivo,apto_conteo_p1,apto_valor_p1,anio_firma,mes_firma,periodo_firma,valor_millones,valor_miles_millones
0,CO1.PCCNTR.770322,000168-2019,ANGIE GERALDINE RINCÓN CALA,1016046149,Cédula de Ciudadanía,"23,500,000.00",Contratación directa,Prestación de servicios,Cerrado,2019-01-29,No,No,ANGIE GERALDINE RINCÓN CALA,1016046149,DOC_1016046149,ANGIE GERALDINE RINCÓN CALA,True,True,True,True,2019,1,2019-01,23.50,0.02
1,CO1.PCCNTR.5481469,4028 DE 2023,Diana Catalina Burbano Delgado,1061699354,Cédula de Ciudadanía,"15,770,000.00",Contratación directa,Prestación de servicios,Cerrado,2023-10-23,No,No,DIANA CATALINA BURBANO DELGADO,1061699354,DOC_1061699354,DIANA CATALINA BURBANO DELGADO,True,True,True,True,2023,10,2023-10,15.77,0.02
2,CO1.PCCNTR.6706615,3429 DE 2024,SERGIO ALEXANDER RAMIREZ YAÑES,79051510,Cédula de Ciudadanía,"31,500,000.00",Contratación directa,Prestación de servicios,Cerrado,2024-08-30,No,No,SERGIO ALEXANDER RAMIREZ YAÑES,79051510,DOC_79051510,SERGIO ALEXANDER RAMIREZ YAÑES,True,True,True,True,2024,8,2024-08,31.50,0.03
3,CO1.PCCNTR.4412417,265 DE 2023,JUAN PABLO CASTILLO VARGAS,1010168538,Cédula de Ciudadanía,"17,700,000.00",Contratación directa,Prestación de servicios,En ejecución,2023-01-16,No,No,JUAN PABLO CASTILLO VARGAS,1010168538,DOC_1010168538,JUAN PABLO CASTILLO VARGAS,True,True,True,True,2023,1,2023-01,17.70,0.02
4,CO1.PCCNTR.8197940,1609-2025,CONSORCIO BELZCON - INVIAS SAI,901978689,NIT,"20,381,638,080.00",Licitación pública Obra Publica,Obra,Modificado,2025-09-12,Si,No,CONSORCIO BELZCON - INVIAS SAI,901978689,DOC_901978689,CONSORCIO BELZCON - INVIAS SAI,True,True,True,True,2025,9,2025-09,"20,381.64",20.38


## 3. Poblaciones para el análisis

In [29]:
# Separamos los contratos aptos para análisis por cantidad
# de los contratos aptos para análisis monetario

df_conteo = df[
    df["apto_conteo_p1"] == True
].copy()

df_valor = df[
    df["apto_valor_p1"] == True
].copy()

print("Contratos para análisis por cantidad:", len(df_conteo))
print("Contratos para análisis por valor:", len(df_valor))

Contratos para análisis por cantidad: 17477
Contratos para análisis por valor: 17244


## 4. Contratación por proveedor

In [30]:
# Calculamos el número de contratos de cada proveedor

proveedores_conteo = (
    df_conteo
    .groupby(
        ["proveedor_id", "proveedor_etiqueta"]
    )
    .agg(
        numero_contratos=("id_contrato", "nunique")
    )
    .reset_index()
)

# Calculamos la participación de cada proveedor

total_contratos = proveedores_conteo["numero_contratos"].sum()

proveedores_conteo["participacion_contratos_pct"] = (
    proveedores_conteo["numero_contratos"]
    / total_contratos
    * 100
)

# Ordenamos de mayor a menor

proveedores_conteo = (
    proveedores_conteo
    .sort_values("numero_contratos", ascending=False)
    .reset_index(drop=True)
)

proveedores_conteo.head(10)

,proveedor_id,proveedor_etiqueta,numero_contratos,participacion_contratos_pct
0,DOC_6759068,RAFAEL HERNAN RODRIGUEZ PRIETO,38,0.22
1,DOC_830126663,RIAC SAS,37,0.21
2,DOC_901245617,AMV CONSULTORES SAS,33,0.19
3,DOC_900899867,INGDECOL S.A.S.,32,0.18
4,DOC_8300929023,UCING SAS,30,0.17
5,DOC_901522208,R&R PROYECTOS DE INGENIERIA SAS,27,0.15
6,DOC_900856332,CYCOV SAS,25,0.14
7,DOC_900868183,PROYECTOS DE INGENIERÍA CUMBRE S.A.S.,24,0.14
8,DOC_9521002,TULCAN L. J. EIVAR,23,0.13
9,DOC_830143733,FORMAS DE INGENIERIA & ARQUITECTURA S.A.S.,21,0.12


In [31]:
# Calculamos el valor total contratado con cada proveedor

proveedores_valor = (
    df_valor
    .groupby(
        ["proveedor_id", "proveedor_etiqueta"]
    )
    .agg(
        numero_contratos=("id_contrato", "nunique"),
        valor_contratado=("valor_del_contrato", "sum")
    )
    .reset_index()
)

# Calculamos la participación sobre el valor total contratado

total_valor = proveedores_valor["valor_contratado"].sum()

proveedores_valor["participacion_valor_pct"] = (
    proveedores_valor["valor_contratado"]
    / total_valor
    * 100
)

# Ordenamos los proveedores por valor contratado

proveedores_valor = (
    proveedores_valor
    .sort_values("valor_contratado", ascending=False)
    .reset_index(drop=True)
)

# Calculamos la participación acumulada

proveedores_valor["participacion_acumulada_pct"] = (
    proveedores_valor["participacion_valor_pct"].cumsum()
)

proveedores_valor.head(10)

,proveedor_id,proveedor_etiqueta,numero_contratos,valor_contratado,participacion_valor_pct,participacion_acumulada_pct
0,NOMBRE_UNION TEMPORAL PEAJES NACIONALES,UNION TEMPORAL PEAJES NACIONALES,1,"1,581,452,956,192.21",4.54,4.54
1,DOC_890922447,CONSTRUCCIONES EL CONDOR S.A,2,"957,773,458,945.00",2.75,7.29
2,NOMBRE_CONSORCIO VIADUCTO CIÉNAGA DEL MAGDALENA,CONSORCIO VIADUCTO CIÉNAGA DEL MAGDALENA,1,"657,176,378,088.00",1.89,9.17
3,NOMBRE_CONSORCIO VARIANTE ORIENTAL 057,CONSORCIO VARIANTE ORIENTAL 057,1,"650,399,558,495.00",1.87,11.04
4,NOMBRE_CONSORCIO SAN FRANCISCO,CONSORCIO SAN FRANCISCO,1,"610,461,362,068.00",1.75,12.79
5,NOMBRE_CONSORCIO VIAL MHC 063,CONSORCIO VIAL MHC 063,1,"593,383,681,529.00",1.70,14.49
6,DOC_890801052,GOBERNACION DE CALDAS,4,"590,653,185,628.00",1.69,16.19
7,NOMBRE_CONSORCIO LEJIA 052,CONSORCIO LEJIA 052,1,"511,743,141,447.00",1.47,17.65
8,NOMBRE_CONSORCIO ANTIOQUIA AL MAR,CONSORCIO ANTIOQUIA AL MAR,1,"496,344,941,711.00",1.42,19.08
9,NOMBRE_CONSORCIO GPS INFRAESTRUCTURA,CONSORCIO GPS INFRAESTRUCTURA,1,"454,775,921,790.00",1.30,20.38


## 5. Indicadores de concentración

In [32]:
# Calculamos CR1, CR5 y CR10 a partir de la participación de los proveedores

cr1_valor = proveedores_valor["participacion_valor_pct"].head(1).sum()
cr5_valor = proveedores_valor["participacion_valor_pct"].head(5).sum()
cr10_valor = proveedores_valor["participacion_valor_pct"].head(10).sum()

# Calculamos el HHI usando las participaciones porcentuales

hhi_valor = (
    proveedores_valor["participacion_valor_pct"] ** 2
).sum()

indicadores_valor = pd.DataFrame({
    "Indicador": ["CR1", "CR5", "CR10", "HHI"],
    "Valor": [
        cr1_valor,
        cr5_valor,
        cr10_valor,
        hhi_valor
    ]
})

indicadores_valor

,Indicador,Valor
0,CR1,4.54
1,CR5,12.79
2,CR10,20.38
3,HHI,85.04


In [33]:
# Calculamos también la concentración según número de contratos

proveedores_conteo = proveedores_conteo.copy()

proveedores_conteo["participacion_acumulada_pct"] = (
    proveedores_conteo["participacion_contratos_pct"].cumsum()
)

cr1_conteo = proveedores_conteo["participacion_contratos_pct"].head(1).sum()
cr5_conteo = proveedores_conteo["participacion_contratos_pct"].head(5).sum()
cr10_conteo = proveedores_conteo["participacion_contratos_pct"].head(10).sum()

hhi_conteo = (
    proveedores_conteo["participacion_contratos_pct"] ** 2
).sum()

indicadores_conteo = pd.DataFrame({
    "Indicador": ["CR1", "CR5", "CR10", "HHI"],
    "Valor": [
        cr1_conteo,
        cr5_conteo,
        cr10_conteo,
        hhi_conteo
    ]
})

indicadores_conteo

,Indicador,Valor
0,CR1,0.22
1,CR5,0.97
2,CR10,1.66
3,HHI,2.31


## 6. Contratación por modalidad

In [34]:
# Calculamos número de contratos por modalidad

modalidad_conteo = (
    df_conteo
    .groupby("modalidad_de_contratacion")
    .agg(
        numero_contratos=("id_contrato", "nunique")
    )
    .reset_index()
)

modalidad_conteo["participacion_contratos_pct"] = (
    modalidad_conteo["numero_contratos"]
    / modalidad_conteo["numero_contratos"].sum()
    * 100
)

modalidad_conteo = modalidad_conteo.sort_values(
    "numero_contratos",
    ascending=False
)

modalidad_conteo

,modalidad_de_contratacion,numero_contratos,participacion_contratos_pct
2,Contratación directa,12261,70.16
7,Mínima cuantía,2276,13.02
0,Concurso de méritos abierto,1156,6.61
6,Licitación pública Obra Publica,669,3.83
9,Selección Abreviada de Menor Cuantía,598,3.42
5,Licitación pública,359,2.05
10,Selección abreviada subasta inversa,84,0.48
8,Seleccion Abreviada Menor Cuantia Sin Manifest...,49,0.28
1,Contratación Directa (con ofertas),21,0.12
3,Contratación régimen especial,3,0.02


In [35]:
# Calculamos el valor contratado por modalidad

modalidad_valor = (
    df_valor
    .groupby("modalidad_de_contratacion")
    .agg(
        valor_contratado=("valor_del_contrato", "sum")
    )
    .reset_index()
)

modalidad_valor["participacion_valor_pct"] = (
    modalidad_valor["valor_contratado"]
    / modalidad_valor["valor_contratado"].sum()
    * 100
)

modalidad_valor = modalidad_valor.sort_values(
    "valor_contratado",
    ascending=False
)

modalidad_valor

,modalidad_de_contratacion,valor_contratado,participacion_valor_pct
6,Licitación pública Obra Publica,"19,729,894,901,367.33",56.61
2,Contratación directa,"7,021,014,576,764.67",20.14
5,Licitación pública,"3,625,157,861,886.92",10.40
0,Concurso de méritos abierto,"2,828,293,400,894.16",8.11
1,Contratación Directa (con ofertas),"704,690,133,455.80",2.02
9,Selección Abreviada de Menor Cuantía,"305,356,099,152.67",0.88
8,Seleccion Abreviada Menor Cuantia Sin Manifest...,"287,041,427,793.00",0.82
7,Mínima cuantía,"238,064,981,431.77",0.68
10,Selección abreviada subasta inversa,"108,998,419,897.07",0.31
3,Contratación régimen especial,"5,455,375,050.19",0.02


## 7. Contratación por tipo de contrato

In [36]:
# Calculamos el número de contratos por tipo de contrato

tipo_conteo = (
    df_conteo
    .groupby("tipo_de_contrato")
    .agg(
        numero_contratos=("id_contrato", "nunique")
    )
    .reset_index()
)

tipo_conteo["participacion_contratos_pct"] = (
    tipo_conteo["numero_contratos"]
    / tipo_conteo["numero_contratos"].sum()
    * 100
)

tipo_conteo = tipo_conteo.sort_values(
    "numero_contratos",
    ascending=False
)

tipo_conteo

,tipo_de_contrato,numero_contratos,participacion_contratos_pct
11,Prestación de servicios,10282,58.83
9,Obra,2700,15.45
10,Otro,1977,11.31
8,Interventoría,1394,7.98
13,Suministros,488,2.79
6,Consultoría,472,2.70
4,Compraventa,66,0.38
3,Comodato,50,0.29
1,Arrendamiento de inmuebles,26,0.15
12,Seguros,12,0.07


In [37]:
# Calculamos el valor contratado por tipo de contrato

tipo_valor = (
    df_valor
    .groupby("tipo_de_contrato")
    .agg(
        valor_contratado=("valor_del_contrato", "sum")
    )
    .reset_index()
)

tipo_valor["participacion_valor_pct"] = (
    tipo_valor["valor_contratado"]
    / tipo_valor["valor_contratado"].sum()
    * 100
)

tipo_valor = tipo_valor.sort_values(
    "valor_contratado",
    ascending=False
)

tipo_valor

,tipo_de_contrato,valor_contratado,participacion_valor_pct
9,Obra,"23,105,077,029,381.88",66.29
10,Otro,"5,252,730,012,364.48",15.07
8,Interventoría,"2,253,083,477,945.66",6.46
5,Concesión,"1,782,943,292,527.21",5.12
11,Prestación de servicios,"1,140,372,378,089.40",3.27
6,Consultoría,"626,701,458,072.60",1.80
0,Acuerdo Marco de Precios,"409,847,336,369.68",1.18
13,Suministros,"163,220,724,817.47",0.47
1,Arrendamiento de inmuebles,"53,534,270,356.11",0.15
4,Compraventa,"41,437,032,258.00",0.12


## 8. Cruces para el análisis

In [38]:
# Relacionamos proveedor y modalidad de contratación

proveedor_modalidad = (
    df_valor
    .groupby(
        [
            "proveedor_id",
            "proveedor_etiqueta",
            "modalidad_de_contratacion"
        ]
    )
    .agg(
        numero_contratos=("id_contrato", "nunique"),
        valor_contratado=("valor_del_contrato", "sum")
    )
    .reset_index()
    .sort_values("valor_contratado", ascending=False)
)

proveedor_modalidad.head(10)

,proveedor_id,proveedor_etiqueta,modalidad_de_contratacion,numero_contratos,valor_contratado
9196,NOMBRE_UNION TEMPORAL PEAJES NACIONALES,UNION TEMPORAL PEAJES NACIONALES,Licitación pública,1,"1,581,452,956,192.21"
5531,DOC_890922447,CONSTRUCCIONES EL CONDOR S.A,Licitación pública Obra Publica,2,"957,773,458,945.00"
9018,NOMBRE_CONSORCIO VIADUCTO CIÉNAGA DEL MAGDALENA,CONSORCIO VIADUCTO CIÉNAGA DEL MAGDALENA,Licitación pública Obra Publica,1,"657,176,378,088.00"
9013,NOMBRE_CONSORCIO VARIANTE ORIENTAL 057,CONSORCIO VARIANTE ORIENTAL 057,Licitación pública Obra Publica,1,"650,399,558,495.00"
8943,NOMBRE_CONSORCIO SAN FRANCISCO,CONSORCIO SAN FRANCISCO,Licitación pública Obra Publica,1,"610,461,362,068.00"
9041,NOMBRE_CONSORCIO VIAL MHC 063,CONSORCIO VIAL MHC 063,Licitación pública Obra Publica,1,"593,383,681,529.00"
5496,DOC_890801052,GOBERNACION DE CALDAS,Contratación directa,4,"590,653,185,628.00"
8788,NOMBRE_CONSORCIO LEJIA 052,CONSORCIO LEJIA 052,Licitación pública Obra Publica,1,"511,743,141,447.00"
8366,NOMBRE_CONSORCIO ANTIOQUIA AL MAR,CONSORCIO ANTIOQUIA AL MAR,Licitación pública Obra Publica,1,"496,344,941,711.00"
8603,NOMBRE_CONSORCIO GPS INFRAESTRUCTURA,CONSORCIO GPS INFRAESTRUCTURA,Licitación pública Obra Publica,1,"454,775,921,790.00"


In [39]:
# Relacionamos proveedor y tipo de contrato

proveedor_tipo = (
    df_valor
    .groupby(
        [
            "proveedor_id",
            "proveedor_etiqueta",
            "tipo_de_contrato"
        ]
    )
    .agg(
        numero_contratos=("id_contrato", "nunique"),
        valor_contratado=("valor_del_contrato", "sum")
    )
    .reset_index()
    .sort_values("valor_contratado", ascending=False)
)

proveedor_tipo.head(10)

,proveedor_id,proveedor_etiqueta,tipo_de_contrato,numero_contratos,valor_contratado
9134,NOMBRE_UNION TEMPORAL PEAJES NACIONALES,UNION TEMPORAL PEAJES NACIONALES,Concesión,1,"1,581,452,956,192.21"
5460,DOC_890922447,CONSTRUCCIONES EL CONDOR S.A,Obra,2,"957,773,458,945.00"
8957,NOMBRE_CONSORCIO VIADUCTO CIÉNAGA DEL MAGDALENA,CONSORCIO VIADUCTO CIÉNAGA DEL MAGDALENA,Obra,1,"657,176,378,088.00"
8952,NOMBRE_CONSORCIO VARIANTE ORIENTAL 057,CONSORCIO VARIANTE ORIENTAL 057,Obra,1,"650,399,558,495.00"
8882,NOMBRE_CONSORCIO SAN FRANCISCO,CONSORCIO SAN FRANCISCO,Obra,1,"610,461,362,068.00"
8979,NOMBRE_CONSORCIO VIAL MHC 063,CONSORCIO VIAL MHC 063,Obra,1,"593,383,681,529.00"
5424,DOC_890801052,GOBERNACION DE CALDAS,Otro,3,"579,514,000,000.00"
8727,NOMBRE_CONSORCIO LEJIA 052,CONSORCIO LEJIA 052,Obra,1,"511,743,141,447.00"
8305,NOMBRE_CONSORCIO ANTIOQUIA AL MAR,CONSORCIO ANTIOQUIA AL MAR,Obra,1,"496,344,941,711.00"
8543,NOMBRE_CONSORCIO GPS INFRAESTRUCTURA,CONSORCIO GPS INFRAESTRUCTURA,Obra,1,"454,775,921,790.00"


## 9. Evolución temporal

In [40]:
# Calculamos la contratación total por año

resumen_anual = (
    df_valor
    .groupby("anio_firma")
    .agg(
        numero_contratos=("id_contrato", "nunique"),
        numero_proveedores=("proveedor_id", "nunique"),
        valor_contratado=("valor_del_contrato", "sum")
    )
    .reset_index()
    .sort_values("anio_firma")
)

resumen_anual

,anio_firma,numero_contratos,numero_proveedores,valor_contratado
0,2017,47,39,"2,460,920,503.48"
1,2018,1070,907,"1,664,193,669,325.44"
2,2019,1253,875,"1,648,932,483,835.79"
3,2020,2037,1707,"4,124,798,592,446.54"
4,2021,2020,1717,"14,451,320,834,221.10"
5,2022,1504,1340,"2,130,403,269,382.45"
6,2023,3871,2943,"1,560,645,661,846.44"
7,2024,2952,2606,"2,559,604,399,877.13"
8,2025,1613,1438,"922,526,077,051.94"
9,2026,877,868,"5,789,081,269,203.37"


In [41]:
# Calculamos la contratación de cada proveedor por año

proveedor_anio = (
    df_valor
    .groupby(
        [
            "anio_firma",
            "proveedor_id",
            "proveedor_etiqueta"
        ]
    )
    .agg(
        numero_contratos=("id_contrato", "nunique"),
        valor_contratado=("valor_del_contrato", "sum")
    )
    .reset_index()
)

# Calculamos la participación del proveedor dentro de cada año

proveedor_anio["participacion_anual_pct"] = (
    proveedor_anio["valor_contratado"]
    / proveedor_anio.groupby("anio_firma")[
        "valor_contratado"
    ].transform("sum")
    * 100
)

proveedor_anio.head()

,anio_firma,proveedor_id,proveedor_etiqueta,numero_contratos,valor_contratado,participacion_anual_pct
0,2017,DOC_15667368,RUBEN DARIO TAMAYO ESPITIA,1,"86,261,910.00",3.51
1,2017,DOC_19387380,JOSE DAVID CASTIBLANCO MENDOZA,1,"13,497,210.00",0.55
2,2017,DOC_37327046,HEDUA MILDRED CARRASCAL ORTIZ,1,"83,002,975.84",3.37
3,2017,DOC_40018660,OLGA MARIA VARGAS HURTADO,1,"11,970,000.00",0.49
4,2017,DOC_4208026,FABIO EDUARDO CELY HERRERA,1,"110,037,150.00",4.47


## 10. Identificación de principales proveedores

In [42]:
# Identificamos los diez proveedores con mayor valor contratado

top10_proveedores = (
    proveedores_valor
    .head(10)["proveedor_id"]
    .tolist()
)

df["grupo_proveedor"] = np.where(
    df["proveedor_id"].isin(top10_proveedores),
    "Top 10",
    "Otros"
)

df["grupo_proveedor"].value_counts()

,count
grupo_proveedor,
Otros,17463
Top 10,14


In [43]:
# Identificamos cuántos proveedores se necesitan para alcanzar el 80% del valor

numero_proveedores_80 = (
    proveedores_valor["participacion_acumulada_pct"]
    .lt(80)
    .sum()
    + 1
)

print(
    "Proveedores necesarios para alcanzar al menos el 80%:",
    numero_proveedores_80
)

Proveedores necesarios para alcanzar al menos el 80%: 216


## 11. Construcción de la base analítica

In [44]:
# Seleccionamos las variables que se utilizarán en el análisis

columnas_analiticas = [
    "id_contrato",
    "referencia_del_contrato",
    "proveedor_id",
    "proveedor_etiqueta",
    "valor_del_contrato",
    "valor_millones",
    "valor_miles_millones",
    "modalidad_de_contratacion",
    "tipo_de_contrato",
    "estado_contrato",
    "fecha_de_firma",
    "anio_firma",
    "mes_firma",
    "periodo_firma",
    "es_grupo",
    "es_pyme",
    "grupo_proveedor",
    "apto_conteo_p1",
    "apto_valor_p1"
]

# Conservamos solamente las columnas existentes

columnas_analiticas = [
    col for col in columnas_analiticas
    if col in df.columns
]

df_analitico_p1 = df[columnas_analiticas].copy()

print("Registros:", df_analitico_p1.shape[0])
print("Variables:", df_analitico_p1.shape[1])

df_analitico_p1.head()

Registros: 17477
Variables: 19


,id_contrato,referencia_del_contrato,proveedor_id,proveedor_etiqueta,valor_del_contrato,valor_millones,valor_miles_millones,modalidad_de_contratacion,tipo_de_contrato,estado_contrato,fecha_de_firma,anio_firma,mes_firma,periodo_firma,es_grupo,es_pyme,grupo_proveedor,apto_conteo_p1,apto_valor_p1
0,CO1.PCCNTR.770322,000168-2019,DOC_1016046149,ANGIE GERALDINE RINCÓN CALA,"23,500,000.00",23.50,0.02,Contratación directa,Prestación de servicios,Cerrado,2019-01-29,2019,1,2019-01,No,No,Otros,True,True
1,CO1.PCCNTR.5481469,4028 DE 2023,DOC_1061699354,DIANA CATALINA BURBANO DELGADO,"15,770,000.00",15.77,0.02,Contratación directa,Prestación de servicios,Cerrado,2023-10-23,2023,10,2023-10,No,No,Otros,True,True
2,CO1.PCCNTR.6706615,3429 DE 2024,DOC_79051510,SERGIO ALEXANDER RAMIREZ YAÑES,"31,500,000.00",31.50,0.03,Contratación directa,Prestación de servicios,Cerrado,2024-08-30,2024,8,2024-08,No,No,Otros,True,True
3,CO1.PCCNTR.4412417,265 DE 2023,DOC_1010168538,JUAN PABLO CASTILLO VARGAS,"17,700,000.00",17.70,0.02,Contratación directa,Prestación de servicios,En ejecución,2023-01-16,2023,1,2023-01,No,No,Otros,True,True
4,CO1.PCCNTR.8197940,1609-2025,DOC_901978689,CONSORCIO BELZCON - INVIAS SAI,"20,381,638,080.00","20,381.64",20.38,Licitación pública Obra Publica,Obra,Modificado,2025-09-12,2025,9,2025-09,Si,No,Otros,True,True


## 12. Validación final

In [45]:
# Revisamos las principales características de la base final

print("Contratos:", df_analitico_p1["id_contrato"].nunique())
print("Proveedores:", df_analitico_p1["proveedor_id"].nunique())
print("Modalidades:", df_analitico_p1["modalidad_de_contratacion"].nunique())
print("Tipos de contrato:", df_analitico_p1["tipo_de_contrato"].nunique())

print(
    "Periodo:",
    df_analitico_p1["fecha_de_firma"].min(),
    "a",
    df_analitico_p1["fecha_de_firma"].max()
)

Contratos: 17477
Proveedores: 8748
Modalidades: 11
Tipos de contrato: 14
Periodo: 2017-11-29 00:00:00 a 2026-08-06 00:00:00


## 13. Exportación de resultados

In [46]:
# Exportamos únicamente la base final preparada para el análisis

df_analitico_p1.to_csv(
    "df_analitico_p1.csv",
    index=False
)

print("Base analítica exportada correctamente.")
print("Archivo: df_analitico_p1.csv")
print("Registros:", df_analitico_p1.shape[0])
print("Variables:", df_analitico_p1.shape[1])

Base analítica exportada correctamente.
Archivo: df_analitico_p1.csv
Registros: 17477
Variables: 19
